> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 2.5 a 2.10 - Pilares da OO

## Exercícios

#### Q1.
Essa lista de exercícios terá como base a classe `Evento` criada em exercícios anteriores. Primeiramente criaremos a classe abstrata `EventoABC` com os métodos de instância abstratos `__str__(self)` e `isConcluido(self)`, indicando que todos as subclasses que dela herdarem devem implementar esses métodos.

`EventoABC` também possui os atributos `_titulo` (string) e `_descricao` (string), cujos valores são recebidos e inicializados no construtor da classe. Note a convenção de nomenclatura indicando o caráter privado desses atributos.

In [13]:
#### Classe EventoABC
from abc import ABC, abstractmethod
class EventoABC(ABC):
    def __init__(self, titulo: str, descricao: str):
        self._titulo = titulo
        self._descricao = descricao
    @abstractmethod
    def __str__(self):
        pass
    @abstractmethod
    def isConcluido(self, *args, **kwargs):
        pass


#### Q2.

Crie a classe `DataHora` que dará suporte ao registro de eventos de calendário.
* A classe possui o atributo de instância `_data_hora` (datetime) privado e um atributo de classe `FORMAT` inicializado com a formatação de string aceito para `_data_hora`, ou seja, `FORMAT = '%d/%m/%Y, %H:%M'`.
* A classe **não possui construtor customizado**. A alteração de seu atributo se dará a partir da propriedade a seguir.
* Crie a `property` `data_hora` para manipular o atributo `_data_hora`.
    * O getter da propriedade deve retornar a data como uma string formatada (`%d/%m/%Y, %H:%M`). Use o atributo `FORMAT`. Consulte o [funcionamento do método `strftime`](https://www.programiz.com/python-programming/datetime/strftime).
    * O setter da propriedade deve receber uma string de data formatada (`%d/%m/%Y, %H:%M`) e implementar um bloco try-except que tenta converter a string em `datetime` e lança um `ValueError` caso a entrada seja inválida. Use o atributo `FORMAT`. Consulte o [funcionamento do método `strptime`](https://www.digitalocean.com/community/tutorials/python-string-to-datetime-strptime).
* Crie o método de instância `isPassado(self)` que avalia se a `_data_hora` é menor que `datetime.now()` (a data e hora atual) e retorna `True` em caso positivo, e `False` caso contrário.
* Crie o método de instância `somaDias(self, num_dias)` que recebe um inteiro `num_dias`, soma esse valor ao atributo interno `_data_hora` e retorna a string formatada do resultado da soma (código dado a seguir).   
```python
data_hora_somada = self._data_hora + datetime.timedelta(days=num_dias)
return data_hora_somada.strftime(FORMAT)
```

Teste a classe `DataHora` com o seguinte código (altere o que for necessário):
```python
# instanciando o objeto
dh = DataHora()

# definindo a data_hora através da propriedade
dh.data_hora = '05/02/2024, 12:30'

## editando a data_hora através da função somaDias
dh.data_hora = dh.somaDias(30)

## imprimindo a data_hora editada e se é passado
print(dh.data_hora, dh.isPassado())
```

In [14]:
#### Classe DataHora
from datetime import datetime, timedelta

class DataHora:
    def __init__(self, data_hora_str: str):
        self._data_hora = datetime.strptime(data_hora_str, "%d/%m/%Y, %H:%M")

    def __str__(self):
        return self._data_hora.strftime("%d/%m/%Y, %H:%M")

    def somaDias(self, dias: int):
        nova_data = self._data_hora + timedelta(days=dias)
        return DataHora(nova_data.strftime("%d/%m/%Y, %H:%M"))

    def isPassado(self):
        return self._data_hora < datetime.now()

    def get_datetime(self):
        return self._data_hora


#### Q3.
Crie a classe `EventoUnico`:
* A classe deve herdar de `EventoABC`.
* Possui o atributo de instância `_data_hora` (classe `DataHora` que criamos previamente).
* Seu construtor deve receber e inicializar os atributos da superclasse, além do valor de `_data_hora` recebido como uma string formatada (`%d/%m/%Y, %H:%M`). Note que para alterar `_data_hora` (objeto tipo `DataHora`), você deve manipular a propriedade interna da classe.
*  Implementa os métodos abstratos da superclasse:
    * Método `isConcluido()` que invoca o método `isPassado()` de `_data_hora` e retorna o seu resultado.
    * Método `__str__` que imprime os atributos do evento na forma `"Evento: _titulo, Data: _data_hora, Descrição: _descricao, Concluido: isConcluido()"`. Note que `isConcluido()` é o método de avaliação implementado. 
* Crie o método de instância `editar_data_hora` que recebe uma string formatada e altera `_data_hora` (através de sua propriedade interna).
    
    
Teste a classe `EventoUnico` com o seguinte código:
```python
# criar evento
evento = EventoUnico('Reunião', 'Sala 302, prédio da esquina', '05/10/2023, 16:30')
print(evento)

# editar data do evento (através da propriedade)
evento.editar_data_hora('05/10/2024, 16:30')
print(evento)
```

In [15]:
#### Classe EventoUnico
class EventoRecorrente(EventoABC):
    def __init__(self, titulo, descricao, data_hora_inicial, data_hora_final, intervalo_repeticao):
        super().__init__(titulo, descricao)
        self.__datas_horas = []  # lista privada de objetos DataHora

        atual = DataHora(data_hora_inicial)
        data_final = DataHora(data_hora_final)
        while atual.get_datetime() <= data_final.get_datetime():
            self.__datas_horas.append(atual)
            atual = atual.somaDias(intervalo_repeticao)

    def isConcluido(self, indice):
        return self.__datas_horas[indice].isPassado()

    def __str__(self):
        texto = ""
        for i, data in enumerate(self.__datas_horas):
            texto += f"Evento: {self._titulo}, Data: {data}, Descrição: {self._descricao}, Concluído: {self.isConcluido(i)}\n"
        return texto.strip()

    def editar_data_hora(self, data_hora_antiga: str, data_hora_nova: str):
        for i, data in enumerate(self.__datas_horas):
            if str(data) == data_hora_antiga:
                self.__datas_horas[i] = DataHora(data_hora_nova)
                break


#### Q3.
Crie a classe `EventoRecorrente`:
* A classe deve herdar de `EventoABC`.
* Possui como atributo próprio uma lista privada de objetos `DataHora` (como você deve nomear o atributo?).
* Seu construtor recebe os atributos da superclasse, além dos atributos `data_hora_inicial` (string formatada), `data_hora_final` (string formatada) e `intervalo_repeticao` (int), sendo o intervalo dado em dias. Preencha a coleção `DataHora` de acordo com o intervalo de repetição fornecido. Dica: crie o objeto `DataHora` inicial e use sua função interna `somaDias` para criar iterativamente as novas instâncias do intervalo até chegar em `DataHora` final. 
*  Implementa os métodos abstratos da superclasse:
    * Método `isConcluido(indice)` que que invoca o método `isPassado()` do elemento `indice` da coleção de objetos `DataHora` e retorna seu resultado. 
    * Método `__str__` que imprime (em um laço) **todos as ocorrências `i` do evento** na forma `"Evento: _titulo, Data: data_hora[i], Descrição: _descricao, Concluido: isConcluido(i)"`. 
* Crie o método `editar_data_hora` que recebe `data_hora_antiga` e `data_hora_nova` e altera o elemento da coleção de objetos `DataHora` que corresponde a `data_hora_antiga` fornecida.    


Teste a classe `EventoRecorrente` com o seguinte código:
```python
# criar evento
eventos = EventoRecorrente(
    'Reunião', 'Sala 302, prédio da esquina', 
    '05/01/2024, 16:30', '05/01/2025, 16:30', 30)

# imprimir eventos
print(eventos)

# editar um dos eventos
eventos.editar_data_hora('05/12/2024, 16:30', '05/12/2024, 11:30')

# imprimir eventos
print(eventos)
```

In [16]:
class EventoRecorrente(EventoABC):
    def __init__(self, titulo, descricao, data_hora_inicial, data_hora_final, intervalo_repeticao):
        super().__init__(titulo, descricao)

        self.__datas_horas = []

        atual = DataHora(data_hora_inicial)
        data_final = DataHora(data_hora_final)

        while atual.get_datetime() <= data_final.get_datetime():
            self.__datas_horas.append(atual)
            atual = atual.somaDias(intervalo_repeticao)

    def isConcluido(self, indice):
        return self.__datas_horas[indice].isPassado()

    def __str__(self):
        texto = ""
        for i, data in enumerate(self.__datas_horas):
            texto += (
                f"Evento: {self._titulo}, "
                f"Data: {data}, "
                f"Descrição: {self._descricao}, "
                f"Concluído: {self.isConcluido(i)}\n"
            )
        return texto.strip()

    def editar_data_hora(self, data_hora_antiga, data_hora_nova):
        for i, data in enumerate(self.__datas_horas):
            if str(data) == data_hora_antiga:
                self.__datas_horas[i] = DataHora(data_hora_nova)
                break

# criar evento
eventos = EventoRecorrente(
    'Reunião', 'Sala 302, prédio da esquina', 
    '05/01/2024, 16:30', '05/01/2025, 16:30', 30)

# imprimir eventos
print(eventos)

# editar um dos eventos
eventos.editar_data_hora('05/12/2024, 16:30', '05/12/2024, 11:30')

# imprimir eventos
print(eventos)

Evento: Reunião, Data: 05/01/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 04/02/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 05/03/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 04/04/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 04/05/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 03/06/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 03/07/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 02/08/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 01/09/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Data: 01/10/2024, 16:30, Descrição: Sala 302, prédio da esquina, Concluído: True
Evento: Reunião, Dat

#### Q4.

Por fim, vamos só ver o polimorfismo em ação. Crie e preencha uma lista de eventos, sendo alguns do tipo `EventoUnico` e outros do tipo `EventoRecorrente`. Sobre essa lista, execute o laço de impressão a seguir:
```python
for evento in lista_eventos: print(evento)
```
A função `print` irá invocar o método especial `__str__` das classes correspondentes dependendo do tipo do objeto recebido. Aí está o polimorfismo :)

In [ ]:
lista_eventos = [
    EventoUnico('Entrevista', 'RH da empresa', '20/10/2024, 09:00'),
    EventoRecorrente('Reunião', 'Sala 302, prédio da esquina',
                     '05/01/2024, 16:30', '05/04/2024, 16:30', 30),
    EventoUnico('Consulta médica', 'Clínica Saúde Total', '12/11/2024, 15:30'),
    EventoRecorrente('Aula de Python', 'Laboratório 5',
                     '10/02/2024, 08:00', '10/06/2024, 08:00', 15)
]

for evento in lista_eventos:
    print(evento)
    print('-' * 80)